# Audio transcription with PyTorch Whisper and Gradio

Upload an audio file, generate a transcript, and download it as CSV (with timestamps) or DOCX. The default `turbo` model is a fast, high-quality choice. Smaller models are available for machines with less compute; `large-v3` is available when you prefer it and have the resources.

## Before you start

- **Run the dependency install cell below before running any other code cell.** If you already see `ModuleNotFoundError: No module named 'whisper'`, run that install cell in this notebook's kernel, wait for it to finish, then restart the kernel and run the cells again from the top.
- **FFmpeg must be installed and available on PATH.** On Windows, install it with `winget install Gyan.FFmpeg` in PowerShell, then restart the notebook kernel. Alternatively, install FFmpeg from [ffmpeg.org](https://ffmpeg.org/download.html).
- Whisper uses PyTorch. The install cell installs the default PyTorch package if needed. For NVIDIA GPU acceleration, install a CUDA-enabled PyTorch build that matches your system from [pytorch.org/get-started/locally](https://pytorch.org/get-started/locally) before running the transcription cell. The code automatically uses CUDA when available and CPU otherwise.
- The first transcription of a model downloads its weights. The `turbo` model needs substantial GPU memory; choose `base` or `small` on resource-limited machines.


In [4]:
%pip install -q openai-whisper gradio python-docx


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 29.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 31.1 MB/s eta 0:00:00


## Transcription and export helpers

The model is loaded once per selected model and reused for later uploads. Each export is written to a unique folder in your system temporary directory; the Gradio file outputs link to those files for download.


In [5]:
import csv
import shutil
import subprocess
import tempfile
import uuid
from functools import lru_cache
from pathlib import Path

import gradio as gr
import torch
try:
    import whisper
except ModuleNotFoundError as error:
    if error.name != "whisper":
        raise
    raise ModuleNotFoundError(
        "OpenAI Whisper is not installed in this notebook kernel. Run the dependency install cell above, "
        "then restart the kernel and rerun the cells."
    ) from error
from docx import Document


MODEL_CHOICES = ["turbo", "base", "small", "medium", "large-v3"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FFMPEG = shutil.which("ffmpeg")

if FFMPEG is None:
    raise RuntimeError(
        "FFmpeg was not found on PATH. Install FFmpeg, restart the notebook kernel, and run this cell again."
    )

print(f"Transcription device: {DEVICE}")
if DEVICE == "cpu":
    print("No CUDA GPU detected. Transcription will run on CPU and may take longer.")


@lru_cache(maxsize=2)
def load_model(model_name):
    """Load and cache a Whisper model on the selected device."""
    if model_name not in MODEL_CHOICES:
        raise ValueError(f"Unsupported model: {model_name}")
    return whisper.load_model(model_name, device=DEVICE)



def transcribe_audio(audio_path, model_name):
    """Transcribe an audio file and return transcript text plus CSV and DOCX paths."""
    if not audio_path:
        raise gr.Error("Upload an audio file before starting transcription.")
    if not Path(audio_path).is_file():
        raise gr.Error("The uploaded audio file could not be found.")

    output_dir = Path(tempfile.gettempdir()) / f"audio_transcript_{uuid.uuid4().hex}"
    output_dir.mkdir(parents=True, exist_ok=False)
    wav_path = output_dir / "audio.wav"

    command = [
        FFMPEG,
        "-nostdin",
        "-y",
        "-i", str(audio_path),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        str(wav_path),
    ]
    try:
        subprocess.run(command, check=True, capture_output=True, text=True)
    except subprocess.CalledProcessError as error:
        details = error.stderr.strip() or "FFmpeg could not extract audio from this file."
        raise gr.Error(f"Could not read audio from the uploaded file: {details}") from error

    model = load_model(model_name)
    result = model.transcribe(
        str(wav_path),
        task="transcribe",
        fp16=(DEVICE == "cuda"),
        verbose=False,
    )
    segments = result.get("segments", [])
    transcript = result.get("text", "").strip()
    if not transcript:
        raise gr.Error("No speech was detected in the uploaded file.")

    csv_path = output_dir / "transcript.csv"
    with csv_path.open("w", newline="", encoding="utf-8-sig") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["start_seconds", "end_seconds", "text"])
        for segment in segments:
            writer.writerow([
                round(segment["start"], 2),
                round(segment["end"], 2),
                segment["text"].strip(),
            ])

    docx_path = output_dir / "transcript.docx"
    document = Document()
    document.add_heading("Audio Transcript", level=1)
    for segment in segments:
        start = float(segment["start"])
        end = float(segment["end"])
        timestamp = f"[{int(start // 60):02d}:{int(start % 60):02d} - {int(end // 60):02d}:{int(end % 60):02d}]"
        document.add_paragraph(f"{timestamp} {segment['text'].strip()}")
    document.save(docx_path)

    return transcript, str(csv_path), str(docx_path)


Transcription device: cuda


## Launch the Gradio app

Run the dependency install cell, then the transcription and export helpers cell, and only then run the app cell below. If a setup cell fails, fix that error before continuing. `share=False` keeps the app local to your machine.


In [ ]:
required_setup = ("gr", "MODEL_CHOICES", "transcribe_audio")
missing_setup = [name for name in required_setup if name not in globals()]
if missing_setup:
    raise RuntimeError(
        "Setup is incomplete. Run the dependency install cell and then the transcription and export helpers cell successfully before launching the app. "
        f"Missing: {', '.join(missing_setup)}"
    )

with gr.Blocks(title="Audio Transcriber") as demo:
    gr.Markdown("# Audio Transcriber\nUpload an audio file to transcribe its speech and download CSV or DOCX files.")
    with gr.Row():
        audio_input = gr.Audio(label="Audio file", sources=["upload"], type="filepath")
        model_input = gr.Dropdown(
            choices=MODEL_CHOICES,
            value="turbo",
            label="Whisper model",
            info="turbo is the fast, high-quality default; base and small use fewer resources.",
        )
    transcribe_button = gr.Button("Transcribe", variant="primary")
    # Gradio 6 removed the show_copy_button argument; a copy button is shown automatically.
    transcript_output = gr.Textbox(label="Transcript", lines=16)
    with gr.Row():
        csv_output = gr.File(label="Download CSV")
        docx_output = gr.File(label="Download DOCX")
    transcribe_button.click(
        fn=transcribe_audio,
        inputs=[audio_input, model_input],
        outputs=[transcript_output, csv_output, docx_output],
    )

demo.launch(share=False, server_port=8001)
